In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

sns.set_theme(context='notebook', style='darkgrid')

In [2]:
def covariance_ellipse(mean, cov, **kwargs):
    eigvals, eigvecs = np.linalg.eigh(cov)

    # Sort largest eigenvalue first
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # Angle of largest eigenvector
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))

    ellipses = []
    for n in range(1, 4):
        
        width = 2 * n * np.sqrt(eigvals[0])
        height = 2 * n * np.sqrt(eigvals[1])
    
        ellipses.append(Ellipse(
            xy=mean.reshape(2,),
            width=width,
            height=height,
            angle=angle,
            fill=False,
            **kwargs
        ))       

    return ellipses

In [3]:
class linear_model:
    def __init__(self, F: np.ndarray , Q: np.ndarray = [[0, 0], [0, 0]]):
        self.F = F 
        self.Q = Q
        
    def __call__(self, x_old: np.ndarray):
        x_new = self.F @ x_old + np.random.multivariate_normal(mean=[0, 0], cov=self.Q, size=1).T
        return x_new

def propogate_uncertainty(input_cov, F, Q):
    return F @ input_cov @ F.T + Q

def compute_gain(H, P, R):
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)
    return K, S

def update_estimate(estimate, measurement, K, H):
    residual = (measurement - H @ estimate)
    update = estimate + K @ residual
    return update, residual

def update_uncertainty(P, R, H, K):
    I = np.eye(*P.shape)
    update = (I - K @ H) @ P @ (I - K @ H).T + K @ R @ K.T
    return update


In [ ]:
# Model Parameters
n = 50000                           # Number of timesteps
dt = 1                             # Step size
F = np.array([[1, dt], [0, 1]])      # system dynamics matrix
Q = np.array([[1e-4, 0], [0, 1e-4]]) # process error
true_model = linear_model(F, Q)      # model including error
est_model = linear_model(F)          # model assuming no error

# Filter Parameters
P0 = np.array([[100, 0], [0, 100]])  # Initial estiamte uncertainty
R = np.array([[9]])                  # Sensor uncertainty
H = np.array([[1, 0]])               # Measurement matrix

# initial state estimate
x0 = np.array([[0], [1]])

# Initialize arrays to contain outputs
true_state = [x0]
x = [x0]
P = [P0]

# debugging information
residuals = []
residual_cov = []

# Run the filter
for i in range(1, n):

    # "truth"
    true_state.append(true_model(true_state[i -1]))
    
    # initial estimate of next state (i - 1 -> i)
    x_init = est_model(x[i - 1])

    # initial uncertainty of next state
    P_init = propogate_uncertainty(P[i - 1], est_model.F, Q)

    # receive a measurement
    measurement = H @ true_state[i] + np.random.multivariate_normal(mean=[0], cov=R)

    # update estimate and uncertainty
    K, S = compute_gain(H, P_init, R)
    x_new, residual = update_estimate(x_init, measurement, K, H)
    P_new = update_uncertainty(P_init, R, H, K)
    
    x.append(x_new)
    P.append(P_new)
    residuals.append(residual.squeeze())
    residual_cov.append(S.squeeze())

# Run the model with no filter, so we can compare results
model_results = [x0]
for i in range(1, n):
    model_results.append(est_model(model_results[i - 1]))

In [ ]:
true_state = np.asarray(true_state)
model_results = np.asarray(model_results)
x = np.asarray(x)

model_error = true_state - model_results
kalman_error = true_state - x

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

t = np.arange(n) * dt
sns.lineplot(x=t, y=true_state[:, 0].squeeze(), ax=axes[0, 0], label="true position")
sns.lineplot(x=t, y=model_results[:, 0].squeeze(), ax=axes[0, 0], label="model estimated position")
sns.lineplot(x=t, y=model_error[:, 0].squeeze(), ax=axes[0, 1], label="model absolute error")

sns.lineplot(x=t, y=true_state[:, 0].squeeze(), ax=axes[1, 0], label="true position")
sns.lineplot(x=t, y=x[:, 0].squeeze(), ax=axes[1, 0], label="filter estimated position")
sns.lineplot(x=t, y=kalman_error[:, 0].squeeze(), ax=axes[1, 1], label="filter error")

plt.tight_layout()
plt.show()

In [ ]:
sns.kdeplot(kalman_error[:, 0].squeeze())